# Proyecto RappiPlus: de datos a decisiones de negocio

**Introducción**


El objetivo de este proyecto es evaluar el desempeño del servicio **RappiPlus** para apoyar **decisiones de negocio basadas en datos**.

Se trabajan con múltiples datasets del negocio:

- **rappiplus_orders_raw.csv** → información de pedidos, precios, descuentos y revenue  
- **rappiplus_catalog.csv** → costos de productos, categorías y proveedores  
- **rappiplus_marketing_spend.csv** → inversión en marketing por canal y país  
- **events / users / user_activity (SQL)** → comportamiento del usuario dentro de la plataforma  
- **experiment_checkout_ui.csv** → resultados de un experimento A/B en el checkout  

El análisis sigue una lógica clara y progresiva:

1. 🔍 Evaluar si podemos confiar en los datos (calidad de datos en Python) 

2. 💰 Analizar si el negocio es rentable (revenue, costos y profit)  

3. 🛒 Entender dónde se pierden los usuarios (funnel de conversión)  

4. 🔁 Evaluar si los usuarios regresan (retención por cohortes)  

5. 🧪 Validar si los cambios generan impacto (test estadístico)  

6. 📊 Comunicar los resultados (dashboard en BI)  

A lo largo del proyecto, se transforman datos en insights para responder preguntas clave del negocio y proponer **recomendaciones accionables**.

---

## 🔹 Paso 1: Cargar y validar la calidad de los datos

---

### 1.1 Carga de datos y vista rápida

**🎯 Objetivo:** Familiarizarte con la estructura de los datasets del negocio antes de analizarlos.

**Instrucciones:**

- Importa las librerías necesarias
- Carga los archivos:
  - `rappiplus_orders_raw.csv`
  - `rappiplus_catalog.csv`
  - `rappiplus_marketing_spend.csv`
- Guarda los DataFrames en:
  - `orders`, `catalog`, `marketing`
- Explora cada dataset.

---

In [1]:
# importar librerías
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns 

In [2]:
# cargar archivos
orders = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_orders_raw.csv')
catalog = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_catalog.csv')
marketing = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_marketing_spend.csv')

In [3]:
# explorar datasets
# tu código aquí

In [4]:
orders.shape

(25100, 12)

In [5]:
catalog.shape

(7, 4)

In [6]:
marketing.shape

(1620, 5)

In [7]:
orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25100 entries, 0 to 25099
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id_pedido           25100 non-null  object 
 1   id_usuario          25100 non-null  object 
 2   fecha_hora_pedido   25100 non-null  object 
 3   pais                24800 non-null  object 
 4   dispositivo         25080 non-null  object 
 5   fuente_referencia   25070 non-null  object 
 6   nombre_producto     25070 non-null  object 
 7   categoria_producto  25020 non-null  object 
 8   cantidad            25050 non-null  float64
 9   precio_unitario     25050 non-null  float64
 10  monto_descuento     25050 non-null  float64
 11  monto_total         25100 non-null  float64
dtypes: float64(4), object(8)
memory usage: 2.3+ MB


---

In [8]:
catalog.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   nombre_producto     7 non-null      object 
 1   categoria_producto  7 non-null      object 
 2   costo_unitario      7 non-null      float64
 3   proveedor           7 non-null      object 
dtypes: float64(1), object(3)
memory usage: 352.0+ bytes


In [9]:
marketing.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1620 entries, 0 to 1619
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   fecha       1620 non-null   object 
 1   pais        1620 non-null   object 
 2   id_campaña  1620 non-null   object 
 3   canal       1519 non-null   object 
 4   gasto       1620 non-null   float64
dtypes: float64(1), object(4)
memory usage: 63.4+ KB


In [10]:
orders.isna().sum()

id_pedido               0
id_usuario              0
fecha_hora_pedido       0
pais                  300
dispositivo            20
fuente_referencia      30
nombre_producto        30
categoria_producto     80
cantidad               50
precio_unitario        50
monto_descuento        50
monto_total             0
dtype: int64

### Revisión y calidad de datos

**🎯 Objetivo:** Detectar y corregir problemas en los datos que puedan afectar el análisis de revenue, costos y rentabilidad.

Se revisan los 3 datasets
- Validar y convertir fechas al formato correcto  
- Revisar variables numéricas (sin negativos o ceros inválidos)  
- Verificar consistencia de montos  
- Eliminar duplicados  
- Revisar variables categóricas 
---

In [11]:
orders_clean = orders.copy()

In [12]:
# Convertir la columna de fecha a tipo datetime
orders_clean['fecha_hora_pedido'] = pd.to_datetime(orders_clean['fecha_hora_pedido'])

In [13]:
orders_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25100 entries, 0 to 25099
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   id_pedido           25100 non-null  object        
 1   id_usuario          25100 non-null  object        
 2   fecha_hora_pedido   25100 non-null  datetime64[ns]
 3   pais                24800 non-null  object        
 4   dispositivo         25080 non-null  object        
 5   fuente_referencia   25070 non-null  object        
 6   nombre_producto     25070 non-null  object        
 7   categoria_producto  25020 non-null  object        
 8   cantidad            25050 non-null  float64       
 9   precio_unitario     25050 non-null  float64       
 10  monto_descuento     25050 non-null  float64       
 11  monto_total         25100 non-null  float64       
dtypes: datetime64[ns](1), float64(4), object(7)
memory usage: 2.3+ MB


In [14]:
marketing_clean = marketing.copy()

In [15]:
# Convertir la columna de fecha a tipo datetime
marketing_clean['fecha'] = pd.to_datetime(marketing_clean['fecha'])

In [16]:
marketing_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1620 entries, 0 to 1619
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   fecha       1620 non-null   datetime64[ns]
 1   pais        1620 non-null   object        
 2   id_campaña  1620 non-null   object        
 3   canal       1519 non-null   object        
 4   gasto       1620 non-null   float64       
dtypes: datetime64[ns](1), float64(1), object(3)
memory usage: 63.4+ KB


In [17]:
#Verificación de datos nulos
orders_clean.isna().sum()

id_pedido               0
id_usuario              0
fecha_hora_pedido       0
pais                  300
dispositivo            20
fuente_referencia      30
nombre_producto        30
categoria_producto     80
cantidad               50
precio_unitario        50
monto_descuento        50
monto_total             0
dtype: int64

In [18]:
#Verificación de datos nulos
marketing_clean.isna().sum()

fecha           0
pais            0
id_campaña      0
canal         101
gasto           0
dtype: int64

In [19]:
#Se consulto a través de .describe datos minimos y máximos, en datos númericos
orders_clean["monto_total"].describe()

count    2.510000e+04
mean     2.072680e+03
std      9.894995e+04
min     -4.926500e+02
25%      1.805075e+02
50%      3.417500e+02
75%      5.185800e+02
max      8.840200e+06
Name: monto_total, dtype: float64

In [20]:
# Filtramos la base para quedarnos solo con valores positivos o cero
orders_clean = orders_clean[orders_clean['cantidad'] >= 0]
orders_clean = orders_clean[orders_clean['monto_total'] >= 0]

In [21]:
# Rellenamos con mediana para evitar sesgos por valores extremos
orders_clean['precio_unitario'] = orders_clean['precio_unitario'].fillna(orders_clean['precio_unitario'].median())
orders_clean['cantidad'] = orders_clean['cantidad'].fillna(orders_clean['cantidad'].median())
orders_clean['monto_descuento'] = orders_clean['monto_descuento'].fillna(0)

In [22]:
#Se verifica si hay datos nulos en datos númericos
orders_clean.isna().sum()

id_pedido               0
id_usuario              0
fecha_hora_pedido       0
pais                  296
dispositivo            20
fuente_referencia      30
nombre_producto        30
categoria_producto     30
cantidad                0
precio_unitario         0
monto_descuento         0
monto_total             0
dtype: int64

In [23]:
#Verificación de "monto_total", para corroborar si hay números negativos
orders_clean["monto_total"].describe()

count    2.504600e+04
mean     2.076331e+03
std      9.905653e+04
min      5.240000e+00
25%      1.804025e+02
50%      3.415600e+02
75%      5.184375e+02
max      8.840200e+06
Name: monto_total, dtype: float64

In [24]:
#Verificación de valores unicos en columna país
orders_clean["pais"].dropna().unique()

array(['Argentina', 'Mexico', 'Colombia', 'mexico', 'colombia',
       'argentina'], dtype=object)

In [25]:
# Convertimos todos los nombres a formato Título (Ej: argentina -> Argentina)
orders_clean['pais'] = orders_clean['pais'].str.title()

In [26]:
orders_clean["pais"].dropna().unique()

array(['Argentina', 'Mexico', 'Colombia'], dtype=object)

In [27]:
# Agregamos la moda en el dato pais 
moda_pais = orders_clean['pais'].mode()[0]
orders_clean['pais'] = orders_clean['pais'].fillna(moda_pais)

In [28]:
# Verificación final de datos nulos en columna pais 
orders_clean.isna().sum()

id_pedido              0
id_usuario             0
fecha_hora_pedido      0
pais                   0
dispositivo           20
fuente_referencia     30
nombre_producto       30
categoria_producto    30
cantidad               0
precio_unitario        0
monto_descuento        0
monto_total            0
dtype: int64

In [29]:
#se reviso datos unicos de las columnas dispositivos, fuente_referencia, nombre_producto, categoría producto
orders_clean["fuente_referencia"].dropna().unique()

array(['organic', 'paid_search', 'social'], dtype=object)

In [30]:
## Verificación inicial de datos nulos 
orders_clean.isna().sum()

id_pedido              0
id_usuario             0
fecha_hora_pedido      0
pais                   0
dispositivo           20
fuente_referencia     30
nombre_producto       30
categoria_producto    30
cantidad               0
precio_unitario        0
monto_descuento        0
monto_total            0
dtype: int64

In [31]:
# Lista de columnas a limpiar
cols_categoricas = ['dispositivo', 'fuente_referencia', 'nombre_producto', 'categoria_producto']

for col in cols_categoricas:
    # Estandarizamos el texto, para evitar duplicados por minúsculas
    orders_clean[col] = orders_clean[col].str.strip().str.capitalize()
    
    # Calculamos la moda de la columna
    moda_valor = orders_clean[col].mode()[0]
    
    # Rellenamos los nulos con esa moda
    orders_clean[col] = orders_clean[col].fillna(moda_valor)

# 2. Verificación final de nulos

orders_clean[cols_categoricas].isna().sum()

dispositivo           0
fuente_referencia     0
nombre_producto       0
categoria_producto    0
dtype: int64

In [32]:
orders_clean.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 25046 entries, 0 to 25099
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   id_pedido           25046 non-null  object        
 1   id_usuario          25046 non-null  object        
 2   fecha_hora_pedido   25046 non-null  datetime64[ns]
 3   pais                25046 non-null  object        
 4   dispositivo         25046 non-null  object        
 5   fuente_referencia   25046 non-null  object        
 6   nombre_producto     25046 non-null  object        
 7   categoria_producto  25046 non-null  object        
 8   cantidad            25046 non-null  float64       
 9   precio_unitario     25046 non-null  float64       
 10  monto_descuento     25046 non-null  float64       
 11  monto_total         25046 non-null  float64       
dtypes: datetime64[ns](1), float64(4), object(7)
memory usage: 2.5+ MB


In [33]:
# Verificación inicial de datos nulos en base catalgo
catalog.isna().sum()

nombre_producto       0
categoria_producto    0
costo_unitario        0
proveedor             0
dtype: int64

In [34]:
#creación de copia para trabajar con la copia
catalog_clean = catalog.copy()

In [35]:
catalog_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   nombre_producto     7 non-null      object 
 1   categoria_producto  7 non-null      object 
 2   costo_unitario      7 non-null      float64
 3   proveedor           7 non-null      object 
dtypes: float64(1), object(3)
memory usage: 352.0+ bytes


In [36]:
# Verificación inicial de datos nulos 
marketing.isna().sum()

fecha           0
pais            0
id_campaña      0
canal         101
gasto           0
dtype: int64

In [37]:
# Crear la copia para trabajar seguro
marketing_clean = marketing.copy()

# Convertir la columna de fecha a tipo datetime
marketing_clean['fecha'] = pd.to_datetime(marketing_clean['fecha'])

In [38]:
marketing_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1620 entries, 0 to 1619
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   fecha       1620 non-null   datetime64[ns]
 1   pais        1620 non-null   object        
 2   id_campaña  1620 non-null   object        
 3   canal       1519 non-null   object        
 4   gasto       1620 non-null   float64       
dtypes: datetime64[ns](1), float64(1), object(3)
memory usage: 63.4+ KB


In [39]:
#Verificación de valores unicos en columna canal
marketing_clean["canal"].dropna().unique()

array(['organic', 'paid_search', 'social'], dtype=object)

In [40]:
# Quitar espacios antes y después de cada nombre de canal

marketing_clean["canal"] = marketing_clean["canal"].str.strip()

In [41]:
##Verificación de valores nulos en columna canal
marketing_clean.isna().sum()

fecha           0
pais            0
id_campaña      0
canal         101
gasto           0
dtype: int64

In [42]:
# Agregamos la moda en el dato canal
moda = marketing_clean["canal"].mode()[0]
marketing_clean["canal"] = marketing_clean["canal"].fillna(moda)

In [43]:
# Verificamos si existen datos nulos después del cambio
marketing_clean.isna().sum()

fecha         0
pais          0
id_campaña    0
canal         0
gasto         0
dtype: int64

In [44]:
#Verificamos que los cambios se hayan guadado
marketing_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1620 entries, 0 to 1619
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   fecha       1620 non-null   datetime64[ns]
 1   pais        1620 non-null   object        
 2   id_campaña  1620 non-null   object        
 3   canal       1620 non-null   object        
 4   gasto       1620 non-null   float64       
dtypes: datetime64[ns](1), float64(1), object(3)
memory usage: 63.4+ KB


In [45]:
#Verificación en la consistencia de montos
# 1. Calculamos el monto esperado según la lógica de negocio
orders_clean['total_esperado'] = (orders_clean['cantidad'] * orders_clean['precio_unitario']) - orders_clean['monto_descuento']


In [46]:
# 2. Identificamos filas donde la diferencia sea significativa 
# Usamos 0.01 por posibles variaciones de decimales (redondeo)
inconsistentes = orders_clean[abs(orders_clean['total_esperado'] - orders_clean['monto_total']) > 0.01]

f"Se encontraron {len(inconsistentes)} registros con montos inconsistentes."


'Se encontraron 1146 registros con montos inconsistentes.'

In [47]:
# 3. Visualizamos las inconsistencias para entender el error
if len(inconsistentes) > 0:
    display(inconsistentes[['cantidad', 'precio_unitario', 'monto_descuento', 'monto_total', 'total_esperado']].head())

,cantidad,precio_unitario,monto_descuento,monto_total,total_esperado
2,2.0,102.99,10.0,195.99,195.98
24,2.0,117.99,0.0,235.99,235.98
35,2.0,467.34,5.0,929.69,929.68
41,2.0,37.77,10.0,65.53,65.54
136,2.0,211.05,15.0,407.09,407.10


In [48]:
# Usamos 0.02 por posibles variaciones de decimales (redondeo) en comparación con la primera clasificación de 0.01.
# Se determina trabajar con los datos cómo se encuentra debido a que no altera la tendencia ni la toma de decisiones 
inconsistentes = orders_clean[abs(orders_clean['total_esperado'] - orders_clean['monto_total']) > 0.02]

f"Se encontraron {len(inconsistentes)} registros con montos inconsistentes."

'Se encontraron 0 registros con montos inconsistentes.'

In [49]:
# Verificamos cuántos registros están exactamente iguales
duplicados = orders_clean.duplicated().sum()
f"Registros duplicados encontrados: {duplicados}"

'Registros duplicados encontrados: 100'

In [50]:
# Eliminación de duplicados
if duplicados > 0:
    orders_clean = orders_clean.drop_duplicates()
    print("Duplicados eliminados exitosamente.")

Duplicados eliminados exitosamente.


**Resumen de Limpieza y Transformación de Datos**

Normalización de Fechas: Se realizó la conversión de tipos de datos de object a datetime en las bases orders_clean y marketing_clean para habilitar el análisis temporal.

**Gestión de Integridad (Valores Nulos y Duplicados):**

Se identificaron y eliminaron 100 registros duplicados en la base orders_clean.

Los valores nulos en variables categóricas (como pais, dispositivo y canal) fueron sustituidos utilizando la moda, tras un proceso de normalización de texto (strip y title case) para asegurar la consistencia de los nombres.

**Depuración de Anomalías Numéricas:**

Se aplicó un filtro para excluir registros con valores negativos o inconsistentes en las columnas de cantidad y monto_total.

Resultado del filtrado: La base final consta de 25,046 registros. Se descartaron 54 datos (0.21% del total inicial de 25,100) por ser considerados ruidos contables o errores de captura que no representaban operaciones reales.

Validación de Consistencia Financiera: Se validó la consistencia de montos encontrando una precisión del 100% bajo un umbral de tolerancia de 0.02, atribuible a redondeos decimales de la fuente original.

In [51]:
orders_clean.to_csv('orders_clean_final.csv', index=False)

In [52]:
marketing_clean.to_csv('marketing_clean_final.csv', index=False)

---
**📦 Exportación**: Una vez finalizada la limpieza, se exportan los datasets para utilizarlos en la última etapa del proyecto.

In [53]:
# exportar datasets
orders.to_csv('orders_clean.csv', index=False)
catalog.to_csv('catalog_clean.csv', index=False)
marketing.to_csv('marketing_clean.csv', index=False)

---

## 🔹 Paso 2: Analizar si el negocio es rentable

### 2.1 Cálculo de KPIs principales

**🎯 Objetivo:** Calcular los indicadores clave del negocio para evaluar ingresos, costos y rentabilidad.

Se usan los 3 datasets (`orders`, `catalog`, `marketing_spend`):

**📊 Parte 1: Rentabilidad del negocio**
- ¿Cuál es el ingreso total (revenue)? 
- ¿Cuál es el costo total? 
- ¿Cuánto se ha invertido en marketing? 
- ¿El negocio es rentable? (calcular profit)  

---

**📈 Parte 2: Comportamiento de ventas**
- ¿Cuál es el ticket promedio por orden? 
- ¿Cuál es la cantidad promedio de productos por orden? 
- ¿Cuál es el producto más vendido?
- ¿Cuánto se ha gastado en marketing por canal? 

In [54]:
# Parte 1 rentabilidad del negocio
# 1. Ingreso Total (Revenue)
ingreso_total = orders_clean['monto_total'].sum()

In [55]:
# 2. Inversión en Marketing
inversion_marketing = marketing_clean['gasto'].sum()

In [56]:
# 3. Costo Total
costo_total = inversion_marketing 

# 4. Rentabilidad (Profit)
profit = ingreso_total - costo_total
roi = (profit / inversion_marketing) * 100 if inversion_marketing > 0 else 0

print(f"Revenue Total: ${ingreso_total:,.2f}")
print(f"Inversión Marketing: ${inversion_marketing:,.2f}")
print(f"Profit (Utilidad): ${profit:,.2f}")
print(f"ROI: {roi:.2f}%")

Revenue Total: $51,966,981.56
Inversión Marketing: $2,871,843.53
Profit (Utilidad): $49,095,138.03
ROI: 1709.53%


In [57]:
#Parte 2 Comportamiento de ventas
#Ticket promedio por orden
ticket_promedio = orders_clean ['monto_total'].mean()

In [58]:
#Cantidad promedio de productos por orden
cant_promedio_producto = orders_clean ['cantidad'].mean()

In [59]:
#Producto más vendido (por cantidad)

Producto_mas_vendido= orders_clean.groupby('nombre_producto')['cantidad'].sum().sort_values(ascending=False).head(1)


In [60]:
#Gasto en marketing por canal
gasto_marketing_canal= marketing_clean.groupby('canal')['gasto'].sum().sort_values(ascending=False)

In [61]:
print(f"Ticket Promedio: ${ticket_promedio:,.2f}")
print(f"Promedio de productos por orden: {cant_promedio_producto:.2f}")
print("\nProducto líder en ventas:")
print(Producto_mas_vendido)
print("\nInversión por Canal de Marketing:")
print(gasto_marketing_canal)

Ticket Promedio: $2,083.18
Promedio de productos por orden: 7.12

Producto líder en ventas:
nombre_producto
Laptop-gaming-16gb    144198.0
Name: cantidad, dtype: float64

Inversión por Canal de Marketing:
canal
paid_search    1040267.31
social          918043.21
organic         913533.01
Name: gasto, dtype: float64


**Respuestas a este paso:**

**Parte 1: Rentabilidad del negocio**

- ¿Cuál es el ingreso total (revenue)? $51,966,982.56
- ¿Cual es el costo total? $2,871,843.5

- ¿Cuánto se ha invertido en marketing? $2,871,843.53
- ¿El negocio es rentable? SI / Profit (utilidad) = $49,095,138.03 

**Parte 2: Comportamiento de ventas**

- ¿Cuál es el ticket promedio por orden? **$2,083.18**
- ¿Cuál es la cantidad promedio de productos por orden? **7.12**
- ¿Cuál es el producto más vendido? **Laptop-gaming-16gb**
- ¿Cuánto se ha gastado en marketing por canal?
- 
            **paid_search**    **1040267.31**   
            **social**          **918043.21**   
            **organic**         **913533.01** 


---

## 🔹 Paso 3: Entender dónde se pierden los usuarios (funnel de conversión)

**🎯 Objetivo:** Analizar el comportamiento de los usuarios para identificar en qué etapa del proceso se pierden.


⚙️**Conexión a la base de datos**:  
Se ejecuta la línea de configuración para conectar con la base de datos y aplicar consultas SQL en la tabla **events**.

---

**📊 Parte 1: Construcción del funnel**
- ¿Cuántos usuarios llegan a cada etapa del funnel?  
- Se calcula el número de usuarios únicos por `nombre_evento`  
- Se ordenan los eventos según el flujo del usuario  

---

**📉 Parte 2: Análisis de conversión**
- Se calcula la tasa de conversión entre cada paso del funnel  
- Se identifica en qué etapa se pierde la mayor cantidad de usuarios  
- ¿Cuál es la tasa de conversión final?
---

In [62]:
import pandas as pd
from sqlalchemy import create_engine

# ======================
# Conexión (NO modificar)
# ======================
db_config = {
    'user': 'practicum_student',
    'pwd': 'QnmDH8Sc2TQLvy2G3Vvh7',
    'host': 'yp-trainers-practicum.cluster-czs0gxyx2d8w.us-east-1.rds.amazonaws.com',
    'port': 5432,
    'db': 'data-analyst-production-db-en'
}

connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(
    db_config['user'],
    db_config['pwd'],
    db_config['host'],
    db_config['port'],
    db_config['db']
)

engine = create_engine(connection_string, connect_args={'sslmode':'require'})

In [63]:
# Explorar tabla events
# =========================
query_events = '''
SELECT *
FROM events;
'''
events = pd.read_sql(query_events, con=engine)
events.head()

,id_usuario,id_sesion,nombre_evento,timestamp_evento,pais,dispositivo,fuente_referencia,categoria_producto
0,user_6772,6a97f2af-32ae-4186-8c92-04025be1a27b,first_visit,2025-05-17,Colombia,desktop,organic,Moda
1,user_5883,369b767c-1c33-4b2f-a652-c7c0ef92cfc9,add_to_cart,2025-02-23,Mexico,mobile,social,Hogar
2,user_5946,60039041-e78b-474c-87b3-c0b7e9c30708,add_payment_info,2025-05-15,Colombia,desktop,social,Electronica
3,user_827,18252a64-f389-4ef7-9e58-dadad4a3491e,purchase,2025-03-31,Mexico,mobile,social,Moda
4,user_2361,221b364e-cdc5-4668-b698-18d5ba849a67,first_visit,2025-01-22,Argentina,desktop,paid_search,Electronica


In [64]:
#Se revisa valores únicos de la base event, por medio de código SQL
query_usuarios_unicos = '''
SELECT 
    COUNT(DISTINCT id_usuario) as total_usuarios_unicos
FROM events
'''
resultado = pd.read_sql(query_usuarios_unicos, con=engine)
print(resultado)

   total_usuarios_unicos
0                   8000


In [65]:
# Usando el DataFrame que ya tienes cargado, utilizando la configuración python
usuarios_unicos = events['id_usuario'].nunique()
print(f"Total usuarios únicos: {usuarios_unicos}")

Total usuarios únicos: 8000


In [66]:
usuarios_unicos = events['nombre_evento'].nunique()
print(f"Total usuarios únicos: {usuarios_unicos}")

Total usuarios únicos: 6


In [67]:
events["nombre_evento"].dropna().unique()

array(['first_visit', 'add_to_cart', 'add_payment_info', 'purchase',
       'begin_checkout', 'select_item'], dtype=object)

In [68]:
#Se revisa valores únicos en el nombre de evento, de la base event, por medio de código SQL
query_usuarios_unicos = '''
SELECT 
    nombre_evento,
    COUNT(DISTINCT id_usuario) as total_usuarios_unicos
FROM events
GROUP BY nombre_evento
'''
resultado = pd.read_sql(query_usuarios_unicos, con=engine)
print(resultado)

      nombre_evento  total_usuarios_unicos
0  add_payment_info                   6250
1       add_to_cart                   7634
2    begin_checkout                   7208
3       first_visit                   7796
4          purchase                   6240
5       select_item                   7582


In [69]:
# Calcular usuarios únicos por nombre de evento, de la base event, por medio de python

funnel_data = events.groupby('nombre_evento')['id_usuario'].nunique().reset_index()

In [70]:
print (funnel_data)

      nombre_evento  id_usuario
0  add_payment_info        6250
1       add_to_cart        7634
2    begin_checkout        7208
3       first_visit        7796
4          purchase        6240
5       select_item        7582


In [71]:
# Definimos el orden lógico del funnel, por medio de python
orden_logico = ['first_visit', 'select_item', 'add_to_cart', 'begin_checkout', 'add_payment_info','purchase']

# Convertimos la columna 'etapa' en un tipo categórico con el orden definido
funnel_data['nombre_evento'] = pd.Categorical(funnel_data['nombre_evento'], categories=orden_logico, ordered=True)

# Ordenamos el DataFrame
funnel_data_ordenado = funnel_data.sort_values('nombre_evento').reset_index(drop=True)

print("Funnel de Conversión:")
display(funnel_data_ordenado)

Funnel de Conversión:


,nombre_evento,id_usuario
0,first_visit,7796
1,select_item,7582
2,add_to_cart,7634
3,begin_checkout,7208
4,add_payment_info,6250
5,purchase,6240


In [72]:
# PARTE 1: Totales del funnel
# ======================

query_totals = '''
SELECT 
    nombre_evento,
    COUNT(DISTINCT id_usuario) as count
FROM events
WHERE nombre_evento IN ('first_visit', 'select_item', 'add_to_cart', 'begin_checkout', 'add_payment_info','purchase')
GROUP BY nombre_evento
ORDER BY
CASE nombre_evento
    WHEN 'first_visit' THEN 1
        WHEN 'select_item' THEN 2
        WHEN 'add_to_cart' THEN 3
        WHEN 'begin_checkout' THEN 4
        WHEN 'add_payment_info' THEN 5
        WHEN 'purchase' THEN 6
        ELSE 7
    END

'''

totals = pd.read_sql(query_totals, con=engine)
totals

,nombre_evento,count
0,first_visit,7796
1,select_item,7582
2,add_to_cart,7634
3,begin_checkout,7208
4,add_payment_info,6250
5,purchase,6240


In [73]:
# PARTE 2: Conversiones
# ======================

query_conversion = '''

WITH cte_firstv AS (
  SELECT DISTINCT id_usuario
  FROM events
  WHERE nombre_evento = 'first_visit'
),
cte_select AS (
  SELECT DISTINCT id_usuario
  FROM events
  WHERE nombre_evento = 'select_item'
),
cte_card AS (
  SELECT DISTINCT id_usuario
  FROM events
  WHERE nombre_evento = 'add_to_cart'
),
cte_begincheck AS (
  SELECT DISTINCT id_usuario
  FROM events
  WHERE nombre_evento = 'begin_checkout'
),
cte_addpay AS (
  SELECT DISTINCT id_usuario
  FROM events
  WHERE nombre_evento = 'add_payment_info'
),
cte_purchase AS (
  SELECT DISTINCT id_usuario
  FROM events
  WHERE nombre_evento = 'purchase'
)

SELECT
(SELECT COUNT(*) FROM cte_firstv) AS firstv_users,
(SELECT COUNT(*) FROM cte_select) AS select_users,
(SELECT COUNT(*) FROM cte_card) AS card_users,
(SELECT COUNT(*) FROM cte_begincheck) AS begincheck_users,
(SELECT COUNT(*) FROM cte_addpay) AS addpay_users,
(SELECT COUNT(*) FROM cte_purchase) AS purchase_users,

ROUND(((SELECT COUNT(*) FROM cte_firstv)-(SELECT COUNT(*) FROM cte_select)) * 100.0
/ NULLIF((SELECT COUNT(*) FROM cte_firstv),0), 2) AS dropoff_after_first_visit_pct,

ROUND(((SELECT COUNT(*) FROM cte_select) -(SELECT COUNT(*) FROM cte_card)) * 100.0
/ NULLIF((SELECT COUNT(*) FROM cte_select),0), 2) AS dropoff_after_select_pct,

ROUND(((SELECT COUNT(*) FROM cte_card)-(SELECT COUNT(*) FROM cte_begincheck)) * 100.0
/ NULLIF((SELECT COUNT(*) FROM cte_card),0), 2) AS dropoff_after_cart_pct,

ROUND(((SELECT COUNT(*) FROM cte_begincheck)-(SELECT COUNT(*) FROM cte_addpay)) * 100.0
/ NULLIF((SELECT COUNT(*) FROM cte_begincheck),0), 2) AS dropoff_after_begincheck_pct,

ROUND(((SELECT COUNT(*) FROM cte_addpay)-(SELECT COUNT(*) FROM cte_purchase)) * 100.0
/ NULLIF((SELECT COUNT(*) FROM cte_addpay),0), 2) AS dropoff_after_addpay_pct

'''

conversion = pd.read_sql(query_conversion, con=engine)
conversion

,firstv_users,select_users,card_users,begincheck_users,addpay_users,purchase_users,dropoff_after_first_visit_pct,dropoff_after_select_pct,dropoff_after_cart_pct,dropoff_after_begincheck_pct,dropoff_after_addpay_pct
0,7796,7582,7634,7208,6250,6240,2.74,-0.69,5.58,13.29,0.16


**Resumen de la construcción de funnel y Análsis de conversión**

- ¿Cuántos usuarios llegan a cada etapa del funnel?
- Número de usuarios únicos por nombre_evento
- Ordenan los eventos según el flujo del usuario

nombre_evento	count

first_visit	       7796

select_item	       7582

add_to_cart	       7634

begin_checkout     7208

add_payment_info   6250

purchase	       6240


**Análisis de conversión**
- Etapa se pierde la mayor cantidad de usuarios

**Revisión del Fact (F):**
Inconsistencia Detectada: Se identificó un comportamiento anómalo donde el volumen de usuarios que añaden productos al carrito supera por un 0.69% a quienes disparan el evento de selección.

Fuga Real: La pérdida crítica de usuarios ocurre inmediatamente después de la primera visita, donde el 97.26% abandona el sitio sin realizar ninguna acción.

**Revisión del Insight (I):**
Error de Tracking: El valor negativo del -0.69% sugiere un posible error en la implementación de los eventos de seguimiento (tags) del sitio, específicamente en el evento select_item, o la existencia de una ruta de compra que se salta este paso.

Acción Recomendada: Validar con el equipo de desarrollo si el botón de "Añadir al carrito" desde la página principal está omitiendo el registro de "Seleccionar artículo".

## 6.🔹 Paso 4: Evaluar si los usuarios regresan (retención por cohortes)

**🎯 Objetivo:** Analizar la retención de usuarios para entender si regresan después de registrarse.

**Tablas**

- `users` 
- `user_activity` 

---
1. Se identifica la cohorte de cada usuario según el **mes de registro**.


2. Se calcula la retención semanal: cuántos usuarios **se mantienen activos** en cada semana desde su registro.
   - `retenido_w1`: usuarios activos en la semana 1  
   - `retenido_w2`: usuarios activos en la semana 2  
   - `retenido_w3`: usuarios activos en la semana 3  


3. Se calcula el porcentaje de retención para cada semana, dividiendo los usuarios retenidos entre los clientes iniciales de la cohorte:  
   - `semana_1`: porcentaje de usuarios retenidos en la semana 1  
   - `semana_2`: porcentaje de usuarios retenidos en la semana 2  
   - `semana_3`: porcentaje de usuarios retenidos en la semana 3  

Se revisa que la columna de fecha esté en formato correcto (`DATE`).  
Se realiza la conversión usando: `CAST(fecha_registro AS DATE)`

In [74]:
# Explorar tabla users
# =========================
query_users = '''
SELECT *
FROM users;
'''
users = pd.read_sql(query_users, con=engine)
users.head(3)

,id_usuario,fecha_registro,país,dispositivo,tipo_plan
0,user_0,2025-01-29,Mexico,mobile,free
1,user_1,2025-01-07,Mexico,mobile,free
2,user_2,2025-03-12,Argentina,mobile,free


In [75]:
#Validamos si cada cliente tiene más de una fecha en el dataset, de acuerdo al resultado, ningún cliente cuenta con más de una fecha de registro
query_users = '''
SELECT id_usuario,
count(fecha_registro) AS conteo_fecha_registro
FROM users
GROUP BY id_usuario
HAVING count(fecha_registro) > 1

'''
users = pd.read_sql(query_users, con=engine)
users.head(3)

,id_usuario,conteo_fecha_registro


In [76]:
# Explorar tabla user_activity
# =========================
query_user_activity = '''
SELECT *
FROM user_activity;
'''
user_activity = pd.read_sql(query_user_activity, con=engine)
user_activity.head(3)

,id_usuario,fecha_actividad,dias_despues_registro,activo
0,user_0,2025-02-05,7,0
1,user_0,2025-02-12,14,1
2,user_0,2025-02-19,21,1


In [77]:

#Revición de usuarios activos
query_user_activity = '''
SELECT activo,
       COUNT(id_usuario) AS total_usuarios
FROM user_activity
WHERE activo = 1
GROUP BY activo;
'''
user_activity = pd.read_sql(query_user_activity, con=engine)
user_activity.head(3)


,activo,total_usuarios
0,1,13315


In [78]:
#Revición de usuarios activos
query_user_activity = '''
SELECT activo,
       COUNT(id_usuario) AS total_usuarios
FROM user_activity
GROUP BY activo;
'''
user_activity = pd.read_sql(query_user_activity, con=engine)
user_activity.head(3)

,activo,total_usuarios
0,0,18685
1,1,13315


In [79]:
#Revición de usuarios activos, por días de registro
query_user_activity = '''
SELECT activo,
       dias_despues_registro AS dias_registro,
       COUNT(DISTINCT(id_usuario)) AS total_usuarios
FROM user_activity
WHERE dias_despues_registro IN (7,14,21)
AND activo = 1
GROUP BY activo,dias_despues_registro
ORDER BY dias_despues_registro ASC;

'''
user_activity = pd.read_sql(query_user_activity, con=engine)
user_activity.head(3)

,activo,dias_registro,total_usuarios
0,1,7,3360
1,1,14,3355
2,1,21,3350


In [80]:
#Se revisa la fecha inicial de registro
query_users = '''

SELECT id_usuario,
       MIN(fecha_registro) AS cohort_fecha
FROM users
GROUP BY id_usuario,fecha_registro
ORDER BY MIN(fecha_registro) ASC;

'''
users = pd.read_sql(query_users, con=engine)
users.head(3)

,id_usuario,cohort_fecha
0,user_351,2025-01-01
1,user_2273,2025-01-01
2,user_6469,2025-01-01


In [81]:
# Se revisa el cohorte mensual y se agrega las retenciones
query_users = '''
WITH cohortes_uno AS (
    SELECT 
        u.id_usuario,
        u.fecha_registro,
        DATE_TRUNC('month', CAST(u.fecha_registro AS DATE)) AS cohorte_mes,
        ua.dias_despues_registro,
        ua.activo
    FROM users u
    LEFT JOIN user_activity ua ON u.id_usuario = ua.id_usuario
    WHERE ua.fecha_actividad IS NOT NULL
)
SELECT
    TO_CHAR(cohorte_mes, 'YYYY-MM'),
    COUNT(DISTINCT id_usuario) AS usuarios_iniciales,
    COUNT(CASE WHEN dias_despues_registro = 7 AND activo = 1 THEN 1 END) AS retained_w1,
    COUNT(CASE WHEN dias_despues_registro = 14 AND activo = 1 THEN 1 END) AS retained_w2,
    COUNT(CASE WHEN dias_despues_registro = 21 AND activo = 1 THEN 1 END) AS retained_w3
FROM cohortes_uno
GROUP BY cohorte_mes
ORDER BY cohorte_mes;
'''
users = pd.read_sql(query_users, con=engine)
users.head(5)


,to_char,usuarios_iniciales,retained_w1,retained_w2,retained_w3
0,2025-01,1627,697,668,656
1,2025-02,1444,611,609,635
2,2025-03,1636,677,705,690
3,2025-04,1606,680,697,663
4,2025-05,1687,695,676,706


In [82]:
# Se revisa el cohorte mensual y se agrega las retenciones por porcentaje
query_users = '''
WITH cohortes_uno AS (
    SELECT 
        u.id_usuario,
        u.fecha_registro,
        DATE_TRUNC('month', CAST(u.fecha_registro AS DATE)) AS cohorte_mes,
        ua.dias_despues_registro,
        ua.activo
    FROM users u
    LEFT JOIN user_activity ua ON u.id_usuario = ua.id_usuario
    WHERE ua.fecha_actividad IS NOT NULL
),
conteos_retenidos AS (
    SELECT
        cohorte_mes,
        COUNT(DISTINCT id_usuario) AS usuarios_iniciales,
        COUNT(CASE WHEN dias_despues_registro = 7 AND activo = 1 THEN 1 END) AS retenido_w1,
        COUNT(CASE WHEN dias_despues_registro = 14 AND activo = 1 THEN 1 END) AS retenido_w2,
        COUNT(CASE WHEN dias_despues_registro = 21 AND activo = 1 THEN 1 END) AS retenido_w3
    FROM cohortes_uno
    GROUP BY cohorte_mes
)
SELECT 
    TO_CHAR(cohorte_mes, 'YYYY-MM') AS cohorte,
    usuarios_iniciales,
    -- Calculamos porcentajes dividiendo entre el total de la cohorte
    ROUND(retenido_w1::numeric / usuarios_iniciales, 4) * 100 AS semana_1_pct,
    ROUND(retenido_w2::numeric / usuarios_iniciales, 4) * 100 AS semana_2_pct,
    ROUND(retenido_w3::numeric / usuarios_iniciales, 4) * 100 AS semana_3_pct
FROM conteos_retenidos
ORDER BY cohorte_mes;

'''
users = pd.read_sql(query_users, con=engine)
users.head(5)


,cohorte,usuarios_iniciales,semana_1_pct,semana_2_pct,semana_3_pct
0,2025-01,1627,42.84,41.06,40.32
1,2025-02,1444,42.31,42.17,43.98
2,2025-03,1636,41.38,43.09,42.18
3,2025-04,1606,42.34,43.40,41.28
4,2025-05,1687,41.20,40.07,41.85


In [83]:
#Se realiza el cohorte mensual de la base users, esta es una recomendación que me dio Dot, donde combina python
query_users = '''
SELECT id_usuario, fecha_registro
FROM users;
'''
users_df = pd.read_sql(query_users, con=engine)

# Convertir a datetime y extraer mes
users_df['fecha_registro'] = pd.to_datetime(users_df['fecha_registro'])
users_df['cohort_mensual'] = users_df['fecha_registro'].dt.to_period('M')

print(users_df.head(3))

  id_usuario fecha_registro cohort_mensual
0     user_0     2025-01-29        2025-01
1     user_1     2025-01-07        2025-01
2     user_2     2025-03-12        2025-03


In [84]:
#Verificación de cohorte combinando python y SQL
query_cohort_simple = '''
SELECT 
    u.id_usuario,
    u.fecha_registro,
    ua.fecha_actividad,
    ua.activo
FROM users u
LEFT JOIN user_activity ua ON u.id_usuario = ua.id_usuario
ORDER BY u.fecha_registro, ua.fecha_actividad;
'''
cohort_raw = pd.read_sql(query_cohort_simple, con=engine)

# Procesar en Python
cohort_raw['fecha_registro'] = pd.to_datetime(cohort_raw['fecha_registro'])
cohort_raw['fecha_actividad'] = pd.to_datetime(cohort_raw['fecha_actividad'])
cohort_raw['cohort_month'] = cohort_raw['fecha_registro'].dt.to_period('M')

print("Datos de cohorte cargados:")
display(cohort_raw.head())

Datos de cohorte cargados:


,id_usuario,fecha_registro,fecha_actividad,activo,cohort_month
0,user_465,2025-01-01,2025-01-08,0,2025-01
1,user_3450,2025-01-01,2025-01-08,0,2025-01
2,user_5408,2025-01-01,2025-01-08,1,2025-01
3,user_894,2025-01-01,2025-01-08,1,2025-01
4,user_1194,2025-01-01,2025-01-08,1,2025-01


In [85]:
# Análisis de retención por cohortes en Python
# ============================================

# 1. Preparar datos de usuarios y actividad
users_df = cohort_raw.copy()

# 2. Filtrar solo usuarios activos
users_activos = users_df[users_df['activo'] == 1].copy()

# 3. Calcular semanas desde registro
users_activos['semanas_desde_registro'] = (
    (users_activos['fecha_actividad'] - users_activos['fecha_registro']).dt.days // 7
)

# 4. Crear tabla de retención por cohorte
def calcular_retencion_cohortes(df):
    # Agrupar por cohorte mensual y semana
    retencion = df.groupby(['cohort_month', 'semanas_desde_registro'])['id_usuario'].nunique().reset_index()
    
    # Crear tabla pivot
    tabla_retencion = retencion.pivot(
        index='cohort_month', 
        columns='semanas_desde_registro', 
        values='id_usuario'
    ).fillna(0)
    
    return tabla_retencion

# 5. Ejecutar análisis
tabla_retencion = calcular_retencion_cohortes(users_activos)

# 6. Calcular porcentajes de retención
tabla_porcentajes = tabla_retencion.div(tabla_retencion.iloc[:, 0], axis=0) * 100

print("Tabla de Retención (Usuarios Absolutos):")
display(tabla_retencion.head())

print("\nTabla de Retención (Porcentajes):")
display(tabla_porcentajes.head())

Tabla de Retención (Usuarios Absolutos):


semanas_desde_registro,1,2,3,4
cohort_month,,,,
2025-01,697,668,656,671
2025-02,611,609,635,575
2025-03,677,705,690,673
2025-04,680,697,663,652
2025-05,695,676,706,679



Tabla de Retención (Porcentajes):


semanas_desde_registro,1,2,3,4
cohort_month,,,,
2025-01,100.0,95.839311,94.117647,96.269727
2025-02,100.0,99.672668,103.927987,94.108020
2025-03,100.0,104.135894,101.920236,99.409158
2025-04,100.0,102.500000,97.500000,95.882353
2025-05,100.0,97.266187,101.582734,97.697842


 ## Análisis sobre los usuarios que o abandonan la plataforma ##

- C: Contexto
Se analizó la retención de usuarios de RappiPlus dividiéndolos en grupos (cohortes) según el mes de su primera compra, monitoreando su comportamiento durante las primeras 3 semanas.

- F: Hechos (Facts)

Estabilidad: La retención en la Semana 1 se mantiene muy estable entre el 41% y 42% para todas las cohortes.

Resiliencia: No hay una caída drástica hacia la Semana 3; de hecho, la cohorte de 2025-02 muestra un incremento atípico subiendo al 43.98% en su tercera semana.

Volumen: Las cohortes son consistentes en tamaño, con un promedio de 1,600 usuarios nuevos por mes.

- I: Insights

Lealtad Sólida: Retener a más del 40% de los usuarios tras tres semanas es un indicador de que el valor de la suscripción de RappiPlus es claro para el usuario desde el inicio.

Oportunidad en el Onboarding: Dado que el mayor "drop" ocurre entre la semana 0 (adquisición) y la semana 1 (donde perdemos al ~58%), el esfuerzo debe centrarse en los primeros 7 días del usuario para asegurar que realicen su segunda compra rápido.


---

## 🔹 Paso 5: Validar si los cambios generan impacto (test estadístico)

🎯 **Objetivo:** Evaluar si la modificación en la UI del checkout impacta la **tasa de conversión de compra**.

---


1. **Analizar el dataset** `experiment_checkout_ui.csv` para identificar la métrica principal **conversion**.
   - La métrica **conversion** es 1 si el usuario completó la compra, 0 si no.    
2. **Plantear la hipótesis estadística**     
3. **Aplicar el test estadístico adecuado** 
4. **Interpretar el resultado**  


---
Hipótesis estadística
   - **H₀ (Hipótesis nula):**
    **“Ambas variantes de compra se comportan igual”**
     
   - **H₁ (Hipótesis alternativa):**
    **"La variante control cuenta con un mayor número de usuarios que completaron una compra"**
     
   
**Test estadístico:**

 **Prueba z de proporciones (z-test)**

**Nivel de significancia alpha:** 

**= 0.05  es el umbral de significancia**


In [86]:
# tu código aquí
from statsmodels.stats.proportion import proportions_ztest
df = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/experiment_checkout_ui.csv')
df.head(10)



,id_usuario,variante,convirtio,dispositivo,pais,duracion_sesion,timestamp
0,exp_user_0,tratamiento,0,mobile,Argentina,114.41,2025-03-28
1,exp_user_1,tratamiento,0,desktop,Mexico,170.03,2025-01-15
2,exp_user_2,control,1,mobile,Colombia,140.21,2025-03-18
3,exp_user_3,tratamiento,0,mobile,Colombia,151.45,2025-06-03
4,exp_user_4,tratamiento,0,desktop,Mexico,299.96,2025-01-12
5,exp_user_5,control,0,mobile,Mexico,206.70,2025-01-26
6,exp_user_6,control,0,desktop,Mexico,280.11,2025-04-15
7,exp_user_7,tratamiento,0,mobile,Argentina,28.47,2025-03-18
8,exp_user_8,tratamiento,0,desktop,Mexico,287.04,2025-02-05
9,exp_user_9,control,0,mobile,Argentina,258.91,2025-02-11


In [87]:
df["variante"].unique()

array(['tratamiento', 'control'], dtype=object)

In [88]:
# Contar número de exitos y número total de registros
conversiones = df.groupby('variante')['convirtio'].sum()
conversiones

variante
control        779
tratamiento    820
Name: convirtio, dtype: int64

In [89]:
totales = df.groupby('variante')['convirtio'].count()
totales

variante
control        4965
tratamiento    5035
Name: convirtio, dtype: int64

In [90]:
exitos = [conversiones ['control'],conversiones['tratamiento']]
exitos

[779, 820]

In [91]:
observaciones = [totales['control'],totales['tratamiento']]
observaciones

[4965, 5035]

In [92]:
# Aplicar prueba y visualizar resultados
z_stat, p_value = proportions_ztest(exitos , observaciones)
print(f"Estadístico z: {z_stat}")
print(f"Valor p: {p_value}")

Estadístico z: -0.8132782986429474
Valor p: 0.41605851639119995


In [93]:
# Aplicar prueba y visualizar resultados
z_stat, p_value = proportions_ztest(exitos , observaciones)
print(f"Estadístico z: {z_stat}")
print(f"Valor p: {p_value:.4f}")

# Interpretar resultados
alpha = 0.05  # umbral de significancia
if p_value < alpha:
    print("Rechazamos la hipótesis nula: hay evidencia de una diferencia.")
else:
    print("No rechazamos la hipótesis nula: no hay evidencia suficiente de una diferencia.")


Estadístico z: -0.8132782986429474
Valor p: 0.4161
No rechazamos la hipótesis nula: no hay evidencia suficiente de una diferencia.


## Resultado del análisis ##
- **C: Contexto**. Se evaluó el impacto de una nueva interfaz de usuario (UI) en el checkout para mejorar la tasa de conversión de compra.Se comparó el desempeño de los usuarios en la variante de tratamiento frente a los de control.

- **F: Hechos (Facts)Resultado Estadístico:** El análisis arrojó un Valor p de 0.4161 y un Estadístico z de -0.8132.Comparativa: Dado que el Valor p (0.4161) es superior al umbral de significancia establecido (0.05), no se puede rechazar la hipótesis nula ($H_0$).

- **Interpretación: No se detectó una diferencia estadísticamente significativa entre ambas versiones de la interfaz.**

- **I: InsightsNeutralidad del Cambio:** La nueva UI del checkout no generó el impacto esperado en las ventas; el comportamiento de compra se mantuvo similar al de la versión original.

- Decisión Estratégica: Se recomienda no implementar el cambio de forma definitiva por ahora. El esfuerzo de desarrollo debería enfocarse en las etapas iniciales del funnel, donde identificamos que se pierde el 97.26% de los usuarios (entre la visita y la selección de artículos), en lugar de optimizar el checkout que ya es bastante eficiente.


---

## 🔹 Paso 6: Comunicar los resultados (Dashboard en BI)

🎯 **Objetivo**:  
Crear un dashboard que muestre de manera clara y visual los resultados del análisis de ventas, costos, marketing y conversión. 

Se usarán los CSVs limpios del Paso 1:

- `orders_clean.csv`  
- `catalog_clean.csv`  
- `marketing_clean.csv`

---

1️⃣ Preparación de los datos
1. Cargar los CSVs en Power BI o Tableau.
2. Revisar relaciones:
   - `orders.nombre_producto` → `catalog.nombre_producto`
   - `orders.fecha_pedido` → tabla de fechas (crear calendario para análisis temporal)
   - `orders.fecha_pedido` → `dim_fecha.date`
3. Crear columnas calculadas necesarias
4. Crear tabla de fechas para poder calcular comparaciones YTD, YoY o períodos anteriores (`Previous Year`, `Previous Month`).

---

2️⃣ Dashboard 1: Overview Ejecutivo
**KPIs principales a mostrar:**
- Revenue total
- Profit total
- Gasto total en marketing
- Ticket promedio
- Cantidad promedio de productos por orden

**Visualizaciones sugeridas:**
- Tarjetas KPI para revenue, profit y gasto marketing
- Gráfico de líneas: evolución mensual de revenue o profit
- Gráfico de líneas YTD
- Gráfico de barras: revenue y profit por producto o categoría

---

 3️⃣ Dashboard 2: Detalle / Drill-through  
**Objetivo:** Permitir explorar los datos desde el KPI general hasta cada orden o producto.

**Visualizaciones sugeridas:**
- Tabla detallada de órdenes con:
  - producto, cantidad, revenue, cost, profit
  - color condicional (profit negativo en rojo, positivo en verde)
- Gráfico de barras por producto con medida `cantidad vendida`
- Drill-through: seleccionar un producto y ver todos los pedidos relacionados
- Filtros por fecha, categoría de producto, etc

### Resulado del análisis por medio de Tableau###

**Análisis Estratégico RappiPlus (Metodología C-F-I)**

1. Desempeño Financiero y Rentabilidad
- Contexto: Se analizaron los ingresos generados frente a la inversión en marketing y costos de productos para el periodo de enero a junio.

- Hallazgo (Finding): El Ingreso Total asciende a $51,966,982, mientras que el Profit se sitúa en $49,095,138. El gasto en marketing representa apenas el 5.5% de los ingresos totales ($2,871,844).

- Insight: El modelo de RappiPlus es altamente rentable. Existe un margen significativo para incrementar la inversión en marketing (especialmente en canales de búsqueda pagada que hoy representan el 36% del gasto) para capturar una mayor cuota de mercado sin comprometer la utilidad.

2. Dinámica de Producto y Categoría

- Contexto: Evaluación del catálogo de productos para identificar motores de volumen vs. motores de margen.

- Hallazgo (Finding): La Laptop-gaming-16gb es el producto estrella en facturación con $43,429,373 (más del 80% del ingreso total), pero su Ganancia Bruta ($12,770) es proporcionalmente inferior a productos de menor costo como la Blender-xl-red o Vacuum-pro-black que superan los $18,000 en ganancia bruta.

- Insight: Tenemos una dependencia crítica de la categoría Electrónica para generar volumen. Se recomienda una estrategia de cross-selling hacia Hogar y Moda, donde los productos (como licuadoras y aspiradoras) tienen un margen de contribución superior y ayudan a diversificar el riesgo de la cartera.

3. Comportamiento de Ventas y Estacionalidad
- Contexto: Análisis de la tendencia de ventas mensual para identificar picos de demanda.

- Hallazgo (Finding): Se observa un pico extraordinario en febrero con $15,963,635, seguido de una caída pronunciada en marzo y abril, recuperándose hacia mayo y junio.

- Insight: La volatilidad en marzo y abril sugiere una sensibilidad alta a factores externos o cambios en la pauta publicitaria. Debemos estabilizar la frecuencia de compra mediante promociones exclusivas de RappiPlus durante los meses de "valle" para garantizar un flujo de caja constante.



---

## 🚀 Entrega Final

Comparte el acceso a tu Dashboard para revisión.   
Puedes entregar el Dashboard utilizando **Power BI o Tableau**.

Incluye **uno de los siguientes**:

- 🔗 Link público del dashboard publicado en **Power BI Service o Tableau Public / Tableau Cloud**
- 🔗 Link de **Google Drive o OneDrive** con el archivo del proyecto (`.pbix`) y los 3 csvs limpios.


### 📎 Enlace del Dashboard

In [95]:
# (Pega aquí tu link)
# link de power bi o tableau

#https://public.tableau.com/views/Proyectofinal_PatriciaGarcia/OverViewEjecutivo?:language=en-US&publish=yes&:sid=&:redirect=auth&:display_count=n&:origin=viz_share_link
